In [ ]:
# WebSearchTool from openai.
# define worklfow and execute it.

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

In [3]:
#OpenAI Agents SDK includes the following hosted tools:

# The `WebSearchTool` lets an agent search the web.  
# The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
# The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.
# This are the tools provided by openai for use.

In [ ]:
# Search Agent

# passed the web search tool, context_size low means it gets cheaper for us.
# here model settings, tells us that a tool call is required.
# u can pass other things like max tokens, temperature, etc.

INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [5]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several AI agent frameworks have emerged, each offering unique capabilities for developing autonomous systems:

- **LangGraph**: Built on LangChain, LangGraph utilizes graph-based state machines to model agent tasks with explicit control flow, making it suitable for complex, stateful workflows. ([duragraph.ai](https://duragraph.ai/blog/agent-frameworks-2025?utm_source=openai))

- **CrewAI**: Focusing on real-time collaboration, CrewAI enables multiple agents or humans and agents to work together effectively, using Python annotations to define agents, tools, tasks, and processes. ([workos.com](https://workos.com/blog/top-ai-agent-frameworks-and-platforms-in-2025?utm_source=openai))

- **AutoGen**: Designed for multi-agent conversations and collaboration, AutoGen provides robust abstractions for agent communication, state tracking, and custom tool usage at scale. ([getmaxim.ai](https://www.getmaxim.ai/articles/top-5-ai-agent-frameworks-in-2025-a-practical-guide-for-ai-builders/?utm_source=openai))

- **LlamaIndex**: Built around retrieval-augmented generation (RAG), LlamaIndex Agents are ideal for knowledge-heavy tasks that rely on accurate data grounding, excellent for building enterprise copilots and domain-specific assistants. ([getmaxim.ai](https://www.getmaxim.ai/articles/top-5-ai-agent-frameworks-in-2025-a-practical-guide-for-ai-builders/?utm_source=openai))

- **OpenAI Agents**: Natively integrated into the OpenAI ecosystem, these agents offer a simplified, API-first framework to connect large language models with tools and memory, best for developers building fast and secure OpenAI stack applications. ([getmaxim.ai](https://www.getmaxim.ai/articles/top-5-ai-agent-frameworks-in-2025-a-practical-guide-for-ai-builders/?utm_source=openai))

These frameworks are part of a rapidly evolving landscape, with major tech companies like Oracle, Salesforce, and Nvidia actively developing and releasing AI agent solutions. Oracle introduced 22 new Fusion Agentic Applications, integrating AI-driven tools into their cloud services. Salesforce launched the AI Foundry service to accelerate agentic AI research and development. Nvidia unveiled multiple open-source initiatives, including the Nemotron Coalition and NemoClaw, aiming to enhance interoperability in the AI ecosystem. ([itpro.com](https://www.itpro.com/technology/artificial-intelligence/oracle-announces-new-proactive-enterprise-agents-at-ai-world-tour-london?utm_source=openai))

As AI agents become more integrated into enterprise operations, frameworks that offer modularity, scalability, and robust security features are increasingly important. Organizations are advised to assess their specific needs and consider the strengths of each framework to select the most suitable solution for their applications. 

In [ ]:

# Planner Agent.

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# description tells the model, why its populatig these fields.
# why we are asking for reason and query, and asking for reason before query, u allow the model to think thru
# and as a result u get better outputs, just by asking rationale first.
# chain of thoughts
# also gives youo a query that is more coherent with this line of thought/ reasoning.


class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:
# not ran as the cost is high.

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    # result = await Runner.run(planner_agent, message)
    print(result.final_output)

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("divyabaheti101@gmail.com") # Change this to your verified email
    to_email = To("divyabaheti101@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
# Email Agent

INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [ ]:
# Writer Agent, for writing the report.

INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [15]:
# functions for workflow.

async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [16]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

In [17]:
# define workflow and call.

query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print(report)




Starting research...
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
short_summary='In 2025, multiple AI agent frameworks have significantly impacted autonomous AI development, with key players like LangChain, Nvidia, and Oracle leading the innovation. As companies increasingly rely on these frameworks, concerns about security and governance also rise, necessitating a focus on monitoring agent behavior to mitigate risks.' markdown_report="# Latest AI Agent Frameworks in 2025\n\n## Table of Contents\n1. [Introduction](#introduction)\n2. [Overview of AI Agent Frameworks](#overview-of-ai-agent-frameworks)\n   - 2.1. LangChain\n   - 2.2. AutoGPT\n   - 2.3. CrewAI\n   - 2.4. LangGraph\n   - 2.5. AutoGen\n   - 2.6. OpenAI Agents SDK\n   - 2.7. Nvidia and NemoClaw\n3. [Enterprise Applications of AI Agent Frameworks](#enterprise-applications-of-ai-agent-frameworks)\n   - 3.1. Oracle’s Fusi